# LCEL chains and PydanticOutputParser

This notebook isolates structured LCEL mechanics: `prompt | llm | parser`, parser failure, and a repair pass.

In [ ]:
%pip install -q "langchain-core==1.5.1" "pydantic==2.13.4"

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from pydantic import BaseModel


class QueryRewriteResult(BaseModel):
    standalone_query: str


parser = PydanticOutputParser(pydantic_object=QueryRewriteResult)
prompt = PromptTemplate.from_template(
    "Rewrite {question} as a standalone query. {format_instructions}"
).partial(format_instructions=parser.get_format_instructions())
json_llm = RunnableLambda(
    lambda _: '{"standalone_query":"remote work approval requirements"}'
)
query_rewrite_chain = prompt | json_llm | parser
print(query_rewrite_chain.invoke({"question": "Who approves it?"}).model_dump())

In [ ]:
from langchain_core.exceptions import OutputParserException

broken_llm = RunnableLambda(lambda _: "not json")
broken_chain = prompt | broken_llm | parser
repair_prompt = PromptTemplate.from_template(
    "Repair this output as valid JSON: {bad_output}. {format_instructions}"
).partial(format_instructions=parser.get_format_instructions())
repair_llm = RunnableLambda(
    lambda _: '{"standalone_query":"repaired standalone query"}'
)
repair_chain = repair_prompt | repair_llm | parser
try:
    result = broken_chain.invoke({"question": "What about it?"})
except OutputParserException as exc:
    result = repair_chain.invoke({"bad_output": exc.llm_output or "not json"})
print(result.model_dump())

In [ ]:
retrieval_chain = RunnableLambda(lambda query: [f"retrieved evidence for: {query}"])
answer_chain = RunnableLambda(lambda state: {"answer": state["documents"][0]})
composed = (
    RunnableLambda(
        lambda values: {**values, "rewrite": query_rewrite_chain.invoke(values)}
    )
    | RunnableLambda(
        lambda state: {
            **state,
            "documents": retrieval_chain.invoke(state["rewrite"].standalone_query),
        }
    )
    | answer_chain
)
print(composed.invoke({"question": "Who approves it?"}))

## Expected output

Three dictionaries are printed: a validated rewrite, a repaired rewrite, and a composed retrieval/answer result.